# 00 — MCP Auth Verification: JWT Foundation

This notebook demonstrates the core mechanism underlying every MCP auth story in this series:

1. **Teleport issues a signed JWT** when a client connects via `tsh mcp connect`.
2. **The backend verifies the JWT** against Teleport’s public JWKS endpoint.
3. **Claims in the verified JWT** — user, roles, cluster — drive identity-aware responses.

Later notebooks layer on AgentCore Gateway, interceptor Lambdas, and Cedar policies — but all of that sits on top of what you build here.

## How It Works

```
Developer
  │  tsh mcp connect <app>
  ↓
Teleport Proxy
  │  Issues JWT: { sub, roles, iss, exp, … }
  │  Injects: Authorization: Bearer <jwt>  (via rewrite.headers)
  ↓
MCP Backend  ← this notebook plays this role
  │  GET https://<cluster>/.well-known/openid-configuration  → jwks_uri
  │  Verifies JWT signature against JWKS public key
  │  Extracts claims → drives identity-aware logic
```

### Key JWT Claims from Teleport

| Claim | Example | Meaning |
|:------|:--------|:--------|
| `sub` | `internal/jeff` | Teleport identity (provider/username) |
| `username` | `jeff` | Human-readable name |
| `roles` | `["mcp-user", "access"]` | Teleport roles assigned to the user |
| `iss` | `https://ellinj.teleport.sh` | Issuing cluster URL |
| `aud` | `mcp+https://…` | Intended audience (the registered app URI) |
| `exp` | Unix timestamp | Expiry — Teleport JWTs are short-lived |

## Step 1: Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install PyJWT cryptography httpx python-dotenv fastmcp uvicorn --quiet

## Step 2: Configure Your Teleport Cluster

Set `TELEPORT_CLUSTER` in your `.env` file (hostname only, no `https://`).

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env', override=True)
TELEPORT_CLUSTER = os.environ.get('TELEPORT_CLUSTER', '')
if not TELEPORT_CLUSTER:
    raise ValueError('Set TELEPORT_CLUSTER in .env (e.g. yourcluster.teleport.sh)')

DISCOVERY_URL = f'https://{TELEPORT_CLUSTER}/.well-known/openid-configuration'
print(f'Cluster       : {TELEPORT_CLUSTER}')
print(f'Discovery URL : {DISCOVERY_URL}')

In [ ]:
.env
.env.example

## Step 3: Define the OIDC Config Helper

A small function that fetches the two values the server needs from Teleport's OIDC discovery endpoint: the issuer URL and the JWKS URI. Keeping this as a named function makes it easy to call at server startup and to test in isolation.

In [ ]:
import httpx

def get_teleport_oidc_config(cluster: str) -> dict:
    """Return issuer and jwks_uri from Teleport's OIDC discovery endpoint."""
    doc = httpx.get(
        f'https://{cluster}/.well-known/openid-configuration',
        timeout=10,
    ).json()
    return {
        'issuer':   doc['issuer'],
        'jwks_uri': doc['jwks_uri'],
    }

# Verify it works before the server starts
config = get_teleport_oidc_config(TELEPORT_CLUSTER)
print(f'issuer   : {config["issuer"]}')
print(f'jwks_uri : {config["jwks_uri"]}')

## Step 4: Build the Identity Response

`build_whoami_response` takes **already-verified claims** and maps them into
the tool response. In Step 5, FastMCP's `JWTVerifier` handles signature
validation, issuer checking, and expiry before this function is ever called —
so by the time claims arrive here, they're guaranteed authentic.

The function does two things:

1. **Extracts the identity** — `sub`, `username`, `roles`, and `iss` come directly from the verified JWT claims
2. **Maps roles to capabilities** — role membership determines `can_read`/`can_write` and which tools the caller may use


In [ ]:
from datetime import datetime, timezone


def build_whoami_response(claims: dict) -> str:
    subject  = claims.get('sub', 'unknown')
    roles    = claims.get('roles', [])
    username = claims.get('username', subject.split('/')[-1] if '/' in subject else subject)
    cluster  = claims.get('iss', '').replace('https://', '')
    exp_dt   = datetime.fromtimestamp(claims.get('exp', 0), tz=timezone.utc)

    can_read  = bool({'mcp-user', 'editor', 'access', 'admin'} & set(roles))
    can_write = bool({'aws-personal-admin', 'editor', 'admin'} & set(roles))

    capabilities = []
    if can_read:
        capabilities.append('read')
    if can_write:
        capabilities.append('write')
    if not capabilities:
        capabilities.append('none')

    roles_str = ', '.join(roles) if roles else 'none'
    caps_str  = ' and '.join(capabilities)
    expires   = exp_dt.strftime('%Y-%m-%d %H:%M UTC')

    return (
        f'You were called by {username} on {cluster} '
        f'with roles [{roles_str}] '
        f'and {caps_str} capabilities (token expires {expires}).'
    )


## Step 5: Start the MCP Server

Three lines set up the security layer:

- **`JWTVerifier`** — fetches Teleport's public keys via JWKS and validates every
  inbound JWT (issuer, signature, expiry). Keys are cached and rotated automatically.
- **`FastMCP(auth=verifier)`** — wires the verifier into the server; unauthenticated
  or invalid requests are rejected before any tool handler runs.
- **`CurrentAccessToken()`** — injects the verified token into the tool handler,
  equivalent to `@AuthenticationPrincipal Jwt jwt` in Spring Security.


In [ ]:
import asyncio
import threading
import uvicorn
from fastmcp import FastMCP
from fastmcp.server.auth.providers.jwt import JWTVerifier
from fastmcp.server.dependencies import CurrentAccessToken

_config = get_teleport_oidc_config(TELEPORT_CLUSTER)
print(f'issuer   : {_config["issuer"]}')
print(f'jwks_uri : {_config["jwks_uri"]}')

verifier = JWTVerifier(
    jwks_uri=_config['jwks_uri'],
    issuer=_config['issuer'],
    # audience omitted — the app URI varies per deployment
)

mcp = FastMCP('teleport-jwt-demo', auth=verifier)


@mcp.tool()
def whoami(token=CurrentAccessToken()) -> str:
    """Returns the Teleport-verified identity of the caller."""
    return build_whoami_response(token.claims)


MCP_PORT = 9999
_uv_server = uvicorn.Server(uvicorn.Config(mcp.http_app(transport='streamable-http'), port=MCP_PORT, log_level='warning'))
_thread = threading.Thread(target=asyncio.run, args=(_uv_server.serve(),), daemon=True)
_thread.start()
print(f'MCP server running on http://localhost:{MCP_PORT}')


## Step 6: Wire the Teleport Agent

The MCP server is running on `localhost:9999`. Teleport needs an app entry in
`teleport.yaml` that points at it — this is what lets `tsh mcp connect` reach
the server and inject the JWT.

The entry below ships with this repo and should already be present. No edits
needed unless you changed the port.

```yaml
app_service:
  enabled: true
  apps:
    - name: identity-aware-mcp
      uri: "mcp+http://localhost:9999/mcp"
      rewrite:
        headers:
          - "Authorization: Bearer {{internal.id_token}}"
```

If the Teleport agent is not running yet:

```bash
teleport start --config teleport.yaml
```

Set `APP_NAME` below to match the `name:` field in your `teleport.yaml`.


In [ ]:
APP_NAME = 'identity-aware-mcp'  # must match name: in teleport.yaml

print(f'App name    : {APP_NAME}')
print(f'Backend URI : mcp+http://localhost:{MCP_PORT}/mcp')
print()
print('Connect with:')
print(f'  tsh mcp connect {APP_NAME}')


## Step 7: Connect via tsh mcp connect

Sends an `initialize` → `tools/list` → `tools/call whoami` sequence through
Teleport. The JWT is injected by the `rewrite.headers` rule in `teleport.yaml`
and validated by `JWTVerifier` before `whoami` runs.


In [ ]:
import subprocess, json as _json

msgs = [
    '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"notebook","version":"0.0.1"}}}',
    '{"jsonrpc":"2.0","id":2,"method":"tools/list","params":{}}',
    '{"jsonrpc":"2.0","id":3,"method":"tools/call","params":{"name":"whoami","arguments":{}}}',
]

result = subprocess.run(
    ['tsh', 'mcp', 'connect', APP_NAME],
    input='\n'.join(msgs) + '\n',
    capture_output=True, text=True, timeout=30,
)

for line in result.stdout.splitlines():
    try:
        print(_json.dumps(_json.loads(line), indent=2))
    except Exception:
        pass

if result.returncode != 0 and result.stderr:
    errors = [l for l in result.stderr.splitlines() if 'level":"error"' in l or 'ERROR' in l]
    for e in errors:
        print('ERROR:', e)


In [ ]:
# Run this cell to stop the server, then re-run the cell above to restart it
_uv_server.should_exit = True
_thread.join(timeout=3)
print('MCP server stopped')
